# Eligible Amount Forecasting

**Objective:** Generate 12-month forecasts for active accounts using historical Eligible Amount balances.

**Model:** XGBoost Regressor (segmented) + RandomForest for volatile segment  
**Forecast Horizon:** 12 Months  
**Forecast Level:** ACCOUNT_ID

## Executive Summary

This notebook forecasts monthly Eligible Amount for each active account over a 12-month horizon using a segmentation-based modeling strategy.

### Business Purpose
- Provide a forward-looking view of expected eligible balances
- Support planning and performance discussions
- Produce structured outputs consumable directly by reporting tools

### What This Notebook Delivers
- Segment-aware validation metrics on recent unseen months
- Account-level forecasts for the next 12 months using segment-specific model routing
- Confidence bounds around forecast values
- Export files for operational and management reporting, including model performance fields for BI

In [1]:
# Import pandas for data manipulation and analysis
import pandas as pd
# Import numpy for numerical computing and array operations
import numpy as np

# Import XGBoost regressor for gradient boosting predictions
from xgboost import XGBRegressor
# Import RandomForest regressor for ensemble-based predictions (volatile segments)
from sklearn.ensemble import RandomForestRegressor
# Import error metrics for model evaluation
from sklearn.metrics import mean_absolute_percentage_error, r2_score

# Import matplotlib for creating plots and visualizations
import matplotlib.pyplot as plt
# Import seaborn for statistical data visualization
import seaborn as sns


## Data Intake and Portfolio Scope

The first preparation steps ensure model inputs represent the real production population.

### What these steps do
- Load source data and convert assignment dates to monthly time format
- Remove incomplete reporting months to avoid partial period bias
- Keep only currently active accounts so the forecast reflects the current portfolio
- Confirm date coverage and account counts before modeling begins

In [ ]:
# Import Snowflake connector library for database connectivity
import snowflake.connector

# Authenticate via SSO browser login (no password stored in code)
# Opens browser for secure authentication without storing credentials
conn = snowflake.connector.connect(
    user="*****************",
    account="************",
    authenticator="***********",     # Opens browser SSO login, no password needed
    warehouse="**************",
    database="*********",
    schema="*************",
    role="************"
)

# Query: Join LDP monthly data with most recent NCUA institutional metrics for each period
# This ensures each account-month has the latest quarterly NCUA filing data available at that time
sql = """
WITH NCUA_MATCH AS (
    -- Left join NCUA metrics to LDP data; take the most recent filing per account-month
    SELECT
        B.ACCOUNT_NAME,
        B.ACCOUNT_ID,
        B.ASSIGNMENT_DATE,
        B.ELIGIBLE_AMOUNT,
        B.PROTECTED_AMOUNT,
        B.ELIGIBLE_COUNT,
        C.CURRENT_MEMBER_COUNT,
        C.TOTAL_ASSET_AMOUNT,
        C.AVERAGE_TOTAL_ASSET_AMOUNT,
        C.TOTAL_LOAN_LEASE_AMOUNT,
        C.TOTAL_SHARE_AND_DEPOSIT_AMOUNT,
        C.FEE_INCOME_AMOUNT,
        C.OTHER_OPERATING_INCOME_AMOUNT,
        C.NON_INTEREST_EXPENSE_AMOUNT,
        C.NET_INCOME_LOSS_AMOUNT,
        C.NET_WORTH_AMOUNT,
        C.FULL_TIME_EMPLOYEE_COUNT,
        C.PART_TIME_EMPLOYEE_COUNT,
        C.FULL_TIME_EQUIVALENT_EMPLOYEE_COUNT,
        C.ANNUALIZED_ASSET_GROWTH,
        C.ANNUALIZED_LOAN_GROWTH,
        C.LOAN_TO_SHARE_RATIO,
        C.AVERAGE_LOAN_BALANCE_AMOUNT,
        C.AVERAGE_SHARES_PER_MEMBER_AMOUNT,
        C.MEMBER_TO_FULL_TIME_EQUIVALENT_EMPLOYEE_RATIO,
        C.DELINQUENT_LOANS_TO_TOTAL_LOANS_RATIO,
        C.DELINQUENT_LOANS_TO_ASSETS_RATIO,
        C.DELINQUENT_LOANS_TO_NET_WORTH_RATIO,
        C.ANNUALIZED_RETURN_ON_ASSETS,
        C.NET_WORTH_TO_TOTAL_ASSETS,
        C.ANNUALIZED_RETURN_ON_EQUITY,
        C.ANNUALIZED_MEMBER_GROWTH,
        C.ASSETS_PER_MEMBER,
        -- Rank NCUA filings by recency within each account-month; take rank 1 (most recent)
        ROW_NUMBER() OVER (
            PARTITION BY B.ACCOUNT_ID, B.ASSIGNMENT_DATE
            ORDER BY C.CYCLE_DATE DESC
        ) AS RN
    FROM S_CMG_BXU_ANALYST_DB_PROD.LX_PRESENTATION.LXU_AYX001065_D_CUSTOPT_MONTHLY_DATA_DETAIL_TB B
    -- Match account to corporate entity ID
    INNER JOIN CDWP_DB_PROD.USER_NCUA.CORPORATE_ORGANIZATION_DIM A
        ON B.ACCOUNT_ID = A.CONTRACT_NUMBER
    -- Attach NCUA metrics where filing date <= month being evaluated (no forward-looking data)
    LEFT JOIN CDWP_DB_PROD.USER_NCUA.NCUA_METRICS_AND_RATIOS_FACT C
        ON A.CORPORATE_ORGANIZATION_HKEY = C.CORPORATE_ORGANIZATION_HKEY
        AND C.CYCLE_DATE <= B.ASSIGNMENT_DATE
    WHERE B.ELIGIBLE_AMOUNT IS NOT NULL
)
SELECT * FROM NCUA_MATCH WHERE RN = 1
"""

# Execute query and load results into pandas DataFrame
cur = conn.cursor()
cur.execute(sql)
# Snowflake connector converts query result directly to pandas DataFrame
df = cur.fetch_pandas_all()
# Close Snowflake connection after data is fetched
conn.close()

# Display shape: (number of rows, number of columns)
print(df.shape)
# Preview first 5 rows of data to inspect structure
df.head()


In [3]:
# Convert ASSIGNMENT_DATE string column to datetime format for time-based operations
df["ASSIGNMENT_DATE"] = pd.to_datetime(df["ASSIGNMENT_DATE"])

# Find earliest and latest dates in dataset to inspect time coverage
raw_min_date = df["ASSIGNMENT_DATE"].min()
raw_max_date = df["ASSIGNMENT_DATE"].max()

# Display date range
print(f"Raw Min Date: {raw_min_date}")
print(f"Raw Max Date: {raw_max_date}")


Raw Min Date: 2013-01-01 00:00:00
Raw Max Date: 2026-07-01 00:00:00


In [4]:
# Remove the most recent two months to avoid partial-period bias
# Most recent data is typically still accumulating (incomplete reporting)
# Convert dates to period format (YYYY-MM) and get unique months sorted
month_index = (
    df["ASSIGNMENT_DATE"]
    .dt.to_period("M")
    .drop_duplicates()
    .sort_values()
)

# Identify the last two months as incomplete (still accumulating data)
incomplete_periods = month_index.tail(2)  # Last two months
# Convert periods back to timestamps for filtering
incomplete_months = incomplete_periods.dt.to_timestamp()

print(f"Excluding incomplete months: {list(incomplete_months)}")

# Filter out the incomplete months from the dataframe
df = df[
    ~df["ASSIGNMENT_DATE"].dt.to_period("M").isin(incomplete_periods)
].copy()

print(f"Post-filter max date: {df['ASSIGNMENT_DATE'].max()}")


Excluding incomplete months: [Timestamp('2026-06-01 00:00:00'), Timestamp('2026-07-01 00:00:00')]
Post-filter max date: 2026-05-01 00:00:00


In [5]:
# Filter to currently active accounts only
# This ensures the forecast reflects the live portfolio (accounts with recent activity)
# Get the latest month in the dataset
latest_month = df["ASSIGNMENT_DATE"].max()

# Find all accounts that have data in the latest month (these are active accounts)
active_accounts = (
    df.loc[
        df["ASSIGNMENT_DATE"] == latest_month,
        "ACCOUNT_ID"
    ]
    .unique()
)

# Keep only the history for active accounts; drop accounts no longer in use
df = df[df["ACCOUNT_ID"].isin(active_accounts)].copy()

print(f"Active Accounts: {len(active_accounts):,}")


Active Accounts: 895


In [6]:
# Confirm number of unique active accounts in the dataset
print(f"Active Accounts: {df['ACCOUNT_ID'].nunique():,}")

# Aggregate detail rows to account-month level
# Sum ELIGIBLE_AMOUNT across all detail rows; use 'first' for institutional metrics (same across detail rows)
df = (
    df.groupby(
        ["ACCOUNT_ID", "ACCOUNT_NAME", "ASSIGNMENT_DATE"],
        as_index=False
    )
    .agg(
        ELIGIBLE_AMOUNT=("ELIGIBLE_AMOUNT", "sum"),  # Sum across detail records
        TOTAL_ASSET_AMOUNT=("TOTAL_ASSET_AMOUNT", "first"),  # NCUA institutional metrics (unchanged per month)
        CURRENT_MEMBER_COUNT=("CURRENT_MEMBER_COUNT", "first"),
        FULL_TIME_EQUIVALENT_EMPLOYEE_COUNT=("FULL_TIME_EQUIVALENT_EMPLOYEE_COUNT", "first"),
        ASSETS_PER_MEMBER=("ASSETS_PER_MEMBER", "first"),
        LOANS_TO_ASSETS=("LOAN_TO_SHARE_RATIO", "first"),  # Loan-to-share ratio as proxy for portfolio risk
        DELINQUENT_LOANS_TO_TOTAL_LOANS_RATIO=("DELINQUENT_LOANS_TO_TOTAL_LOANS_RATIO", "first"),
        ANNUALIZED_ASSET_GROWTH=("ANNUALIZED_ASSET_GROWTH", "first")
    )
)

print(f"Data range after aggregation: {df['ASSIGNMENT_DATE'].min()} to {df['ASSIGNMENT_DATE'].max()}")


Active Accounts: 895
Data range after aggregation: 2024-01-01 00:00:00 to 2026-05-01 00:00:00


## Feature Engineering Approach

Raw monthly balances are transformed into model features that capture memory and seasonality.

### Feature groups used
- **Lag features** at multiple lookbacks to capture prior account behavior
- **Rolling averages** to smooth short-term volatility
- **Calendar features** such as month and quarter to capture repeating seasonal patterns
- **NCUA institutional features** to capture credit union size, growth, and risk profile

### Control for model integrity
Lag and rolling features are created using prior observations only, which prevents future data leakage.
NCUA columns are carried forward (forward-filled per account) to fill months where quarterly data is unavailable.

In [7]:
# Define NCUA institutional/regulatory features to carry forward through time
# These are quarterly metrics; we'll forward-fill to monthly to avoid NA values between filings
NCUA_COLS = [
    "TOTAL_ASSET_AMOUNT",              # Size metric
    "CURRENT_MEMBER_COUNT",            # Member growth indicator
    "FULL_TIME_EQUIVALENT_EMPLOYEE_COUNT",  # Operational scale
    "ASSETS_PER_MEMBER",               # Efficiency metric
    "LOANS_TO_ASSETS",                 # Risk/portfolio composition
    "DELINQUENT_LOANS_TO_TOTAL_LOANS_RATIO",  # Credit quality
    "ANNUALIZED_ASSET_GROWTH"          # Growth trajectory
]

def create_features(df):
    """Build time-series features for forecasting without future data leakage."""
    # Sort by account and date to ensure correct lag calculations
    df = df.sort_values(["ACCOUNT_ID", "ASSIGNMENT_DATE"])

    # Group by account for per-account feature engineering
    grp = df.groupby("ACCOUNT_ID")

    # Create lag features: historical eligible amounts at various lookbacks
    # These capture account momentum and prior behavior
    df["lag_1"]  = grp["ELIGIBLE_AMOUNT"].shift(1)   # 1 month prior
    df["lag_3"]  = grp["ELIGIBLE_AMOUNT"].shift(3)   # 3 months prior
    df["lag_6"]  = grp["ELIGIBLE_AMOUNT"].shift(6)   # 6 months prior
    df["lag_12"] = grp["ELIGIBLE_AMOUNT"].shift(12)  # 12 months prior (year-over-year)

    # Create rolling average features: smoothed trends to reduce short-term noise
    # Use shift(1) first to prevent future leakage (use only past data in rolling window)
    df["rolling_3"]  = grp["ELIGIBLE_AMOUNT"].transform(lambda x: x.shift(1).rolling(3).mean())   # 3-month avg
    df["rolling_6"]  = grp["ELIGIBLE_AMOUNT"].transform(lambda x: x.shift(1).rolling(6).mean())   # 6-month avg
    df["rolling_12"] = grp["ELIGIBLE_AMOUNT"].transform(lambda x: x.shift(1).rolling(12).mean())  # 12-month avg

    # Calendar features: capture seasonal patterns that repeat annually
    df["month"]   = df["ASSIGNMENT_DATE"].dt.month    # 1-12
    df["quarter"] = df["ASSIGNMENT_DATE"].dt.quarter  # Q1-Q4

    # Forward-fill NCUA institutional columns: NCUA files quarterly, not monthly
    # Carry the latest available metric forward to fill gaps (most recent = best estimate)
    for col in NCUA_COLS:
        if col in df.columns:
            df[col] = grp[col].transform(lambda x: x.ffill())  # Forward fill per account group

    return df

# Apply feature engineering function to dataframe
df = create_features(df)


In [8]:
# Apply feature engineering function to dataframe
df = create_features(df)

# Ensure all NCUA columns are numeric to avoid XGBoost dtype errors
for col in NCUA_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Remove rows where lag features are missing (these are early months without enough history)
# Only keep rows with sufficient historical data (at least 1 month back for lag_1)
df = df.dropna(subset=["lag_1", "lag_3", "lag_6", "lag_12"]).copy()

# Fill any remaining NCUA nulls with column median (handles accounts with no NCUA match)
# This is a reasonable default for accounts without NCUA data
for col in NCUA_COLS:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

# Display cleaned data shape
print(df.shape)


(13284, 20)


## Validation Design

A time-based holdout window is used to evaluate performance on recent unseen months, with segmentation preserved through prediction and calibration.

### Why this is important
- Training and testing are separated by time
- Segment-specific prediction behavior can be evaluated before full-history retraining
- This closely matches real forecasting conditions
- Reported metrics are more representative of production behavior

In [9]:
# Find latest date in data
latest_date = df["ASSIGNMENT_DATE"].max()

# Calculate test split: go back 6 months
test_split_date = latest_date - pd.DateOffset(months=5)
# Round to first day of month
test_split_date = test_split_date.replace(day=1)

# Split data
train_df = df[df["ASSIGNMENT_DATE"] < test_split_date].copy()
test_df  = df[df["ASSIGNMENT_DATE"] >= test_split_date].copy()

print(f"Latest date: {latest_date}")
print(f"Test split date: {test_split_date}")
print(f"Training set: {train_df.shape[0]:,} rows (through {train_df['ASSIGNMENT_DATE'].max()})")
print(f"Test set: {test_df.shape[0]:,} rows ({test_df['ASSIGNMENT_DATE'].min()} to {test_df['ASSIGNMENT_DATE'].max()})")


Latest date: 2026-05-01 00:00:00
Test split date: 2025-12-01 00:00:00
Training set: 8,140 rows (through 2025-11-01 00:00:00)
Test set: 5,144 rows (2025-12-01 00:00:00 to 2026-05-01 00:00:00)


In [10]:
# Define the feature matrix for all models
# Combined: past behavior (lags, rolling avg), seasonality, and institutional context
# These features will be used to train all segment models and fallback model
features = [
    # Time-series memory features - capture recent account behavior
    "lag_1", "lag_3", "lag_6", "lag_12",
    # Trend features - smooth out short-term volatility
    "rolling_3", "rolling_6", "rolling_12",
    # Seasonal features - capture repeating annual patterns
    "month", "quarter",
    # Institutional/risk context - NCUA regulatory metrics
    "TOTAL_ASSET_AMOUNT",
    "CURRENT_MEMBER_COUNT",
    "FULL_TIME_EQUIVALENT_EMPLOYEE_COUNT",
    "ASSETS_PER_MEMBER",
    "ANNUALIZED_ASSET_GROWTH",
    "LOANS_TO_ASSETS",
    "DELINQUENT_LOANS_TO_TOTAL_LOANS_RATIO",
]


## Model Configuration and Training

The model uses a segmented configuration (size + volatility profiles) with a stable fallback model to balance accuracy and maintainability.

### Why this setup is used
- Accounts are grouped into practical behavioral segments (core, volatile, large)
- Different model families are applied where behavior differs most
- A global fallback protects coverage for sparse or unmatched segment cases
- The approach stays simple enough for repeatable monthly operations

After holdout testing, segment models are retrained on full history before generating future forecasts.

In [11]:
# SEGMENTATION LOGIC: Identify behavioral clusters to apply different models
# Build per-account profile from training data only (prevent test-set leakage)
# Compute median balance (size proxy), mean, and std deviation for each account
account_train_profile = (
    train_df.groupby(["ACCOUNT_ID", "ACCOUNT_NAME"], as_index=False)["ELIGIBLE_AMOUNT"]
    .agg(
        segment_anchor="median",      # Typical balance (size proxy)
        segment_mean="mean",           # Mean balance
        segment_std="std"              # Standard deviation (volatility proxy)
    )
)

# Coefficient of variation: volatility metric independent of account size
# High CV = high unpredictability; Low CV = stable, predictable behavior
# CV = (standard deviation) / (mean) - allows comparison across different-sized accounts
account_train_profile["segment_cv"] = (
    account_train_profile["segment_std"].fillna(0.0)
    / account_train_profile["segment_mean"].replace(0, np.nan)
).fillna(0.0)

# Define threshold quantiles for segmentation
# These are computed from the training data to identify which accounts are "large" or "volatile"
q_size_large = account_train_profile["segment_anchor"].quantile(0.75)  # Top 25% by size
q_vol_high = account_train_profile["segment_cv"].quantile(0.75)        # Top 25% by volatility

def assign_segment(anchor_value, cv_value):
    """Assign account to segment based on size and volatility profiles."""
    # Prioritize size: large accounts get strictest guardrails regardless of volatility
    if anchor_value >= q_size_large:
        return "large"       # Large accounts: prioritize stability and model accuracy
    # Then check volatility: volatile accounts get ensemble model for robustness
    if cv_value >= q_vol_high:
        return "volatile"    # Volatile accounts: use ensemble (RandomForest) for robustness
    # Default segment for normal-sized, stable accounts
    return "core"            # Core accounts: standard XGBoost model (majority of portfolio)

# Assign segment to each account using the assignment function
account_train_profile["segment"] = account_train_profile.apply(
    lambda r: assign_segment(r["segment_anchor"], r["segment_cv"]), axis=1
)

# Create segment mapping (account ID -> segment) for training data
account_segment_map = account_train_profile[["ACCOUNT_ID", "segment"]].copy()
# Merge segment assignment back with training data
train_with_segment = train_df.merge(account_segment_map, on="ACCOUNT_ID", how="left")

# MINIMUM SEGMENT SIZE THRESHOLD: Ensure sufficient training samples per model
# Models with too few samples overfit; consolidate small segments into "core"
# Minimum rows needed for stable model training
min_segment_rows = 250
# Count how many rows are in each segment
segment_counts = train_with_segment["segment"].value_counts()
# Identify segments that fall below minimum threshold
small_segments = segment_counts[segment_counts < min_segment_rows].index.tolist()

# Redistribute small segments to "core" (most robust baseline model)
# This prevents overfitting on sparse segments
if small_segments:
    print(f"Consolidating small segments {small_segments} to 'core' due to low sample count")
    # Update segment assignment for small segments
    account_segment_map.loc[
        account_segment_map["segment"].isin(small_segments), "segment"
    ] = "core"
    # Re-merge with updated segment assignments
    train_with_segment = train_df.merge(account_segment_map, on="ACCOUNT_ID", how="left")


In [12]:
# Update training dataframe with segment assignments
train_df = train_with_segment.copy()

# TRAIN SEGMENT-SPECIFIC MODELS
# Strategy: Apply different algorithms to accounts with different behavior patterns
# Segment order for model training
segment_order = ["core", "volatile", "large"]
# Dictionary to store trained models for each segment
segment_models = {}

# Train a model for each segment
for seg in segment_order:
    # Filter training data to only this segment
    seg_train = train_df[train_df["segment"] == seg].copy()

    # Skip segment if it has too few rows to train a reliable model
    if len(seg_train) < min_segment_rows:
        print(f"Skipping segment '{seg}' due to low row count: {len(seg_train)}")
        continue

    # SELECT MODEL BY SEGMENT - different algorithms for different behaviors
    if seg == "volatile":
        # Volatile accounts: use RandomForest (ensemble, less prone to overfitting on noisy data)
        seg_model = RandomForestRegressor(
            n_estimators=400,      # More trees = better generalization
            max_depth=14,          # Deeper trees to capture nonlinear patterns
            min_samples_leaf=2,    # Leaf size for split decision (lower = more detail)
            random_state=42, n_jobs=-1
        )
    else:
        # Core and large accounts: use XGBoost (gradient boosting, better for stable data)
        seg_model = XGBRegressor(
            objective="reg:squarederror",
            n_estimators=500,      # Boosting iterations
            max_depth=6,           # Tree depth (controls complexity)
            learning_rate=0.03,    # Shrinkage (smaller = slower learning but more stable)
            subsample=0.8,         # Row sampling (prevent overfitting)
            colsample_bytree=0.8,  # Column sampling (feature selection per iteration)
            random_state=42, n_jobs=-1
        )

    # Train the model on this segment's data
    seg_model.fit(seg_train[features], seg_train["ELIGIBLE_AMOUNT"])
    # Store trained model for later prediction
    segment_models[seg] = seg_model
    print(f"Model trained for '{seg}' segment with {len(seg_train):,} rows")

# FALLBACK MODEL: Global model trained on all data
# Use if segment-specific model is unavailable or for unmatched accounts (rare)
# Ensures prediction coverage for any account
fallback_model = XGBRegressor(
    objective="reg:squarederror", n_estimators=500, max_depth=6,
    learning_rate=0.03, subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1
)
# Train fallback model on all training data
fallback_model.fit(train_df[features], train_df["ELIGIBLE_AMOUNT"])
# Set as reference model for feature importance analysis
model = fallback_model

# Print segmentation summary for transparency
print(f"\nSegmentation Summary:")
print(f"  Large threshold: >= {q_size_large:,.2f} median balance")
print(f"  Volatile threshold: CV >= {q_vol_high:.3f}")
print(f"  Models trained: {list(segment_models.keys())}")
print(f"  Global fallback guarantees prediction coverage for any unmatched account")


Model trained for 'core' segment with 4,252 rows
Model trained for 'volatile' segment with 1,889 rows
Model trained for 'large' segment with 1,999 rows

Segmentation Summary:
  Large threshold: >= 5,448,412.38 median balance
  Volatile threshold: CV >= 0.299
  Models trained: ['core', 'volatile', 'large']
  Global fallback guarantees prediction coverage for any unmatched account


In [13]:
# VALIDATION ON HOLDOUT TEST SET
# Route each test account to its segment model; use fallback for any unmatched rows
# Merge segment assignments into test data
test_df = test_df.merge(account_segment_map, on="ACCOUNT_ID", how="left")
# Default unmapped accounts to core segment (shouldn't happen, but defensive coding)
test_df["segment"] = test_df["segment"].fillna("core")
# Initialize predictions column as NaN (will fill with actual predictions)
test_df["Prediction"] = np.nan

# Predict using segment-specific models
# Each segment gets predictions from its trained model
for seg, seg_model in segment_models.items():
    # Create boolean mask for rows belonging to this segment
    seg_mask = test_df["segment"] == seg
    # If there are rows in this segment, predict using segment model
    if seg_mask.any():
        test_df.loc[seg_mask, "Prediction"] = seg_model.predict(test_df.loc[seg_mask, features])

# Fallback for any rows that couldn't be predicted by segment model (edge cases)
# Identify rows with missing predictions
missing_pred_mask = test_df["Prediction"].isna()
# Use fallback model for rows without predictions
if missing_pred_mask.any():
    test_df.loc[missing_pred_mask, "Prediction"] = fallback_model.predict(
        test_df.loc[missing_pred_mask, features]
    )

# CALIBRATION BY SEGMENT: Adjust predictions to match actual total per segment
# If predictions systematically under- or over-estimate, apply segment-level correction
# Bias factor = (actual total) / (predicted total) per segment
# This ensures forecast totals match actuals per segment
segment_calibration = (
    test_df.groupby("segment", as_index=False)
    .agg(actual_sum=("ELIGIBLE_AMOUNT", "sum"), pred_sum=("Prediction", "sum"))
)

# Calculate and clip bias factors to avoid extreme adjustments (±10% band)
# Bias = actual / predicted; clipped to [0.90, 1.10] to prevent over-correction
segment_calibration["bias_factor"] = np.where(
    segment_calibration["pred_sum"].abs() > 1e-9,
    segment_calibration["actual_sum"] / segment_calibration["pred_sum"],
    1.0
).clip(0.90, 1.10)  # Clamp to [0.90, 1.10] to prevent over-correction

# Create lookup dictionary: segment -> calibration factor
segment_bias_factors = dict(
    zip(segment_calibration["segment"], segment_calibration["bias_factor"])
)

# Apply calibration to all predictions
# Multiply each prediction by its segment's bias factor
test_df["Prediction"] = (
    test_df["Prediction"] * test_df["segment"].map(segment_bias_factors).fillna(1.0)
)

# Print calibration summary
print("Segment Calibration Factors (adjustment to match actuals):")
print(segment_calibration[["segment", "bias_factor"]].sort_values("segment"))
print(f"Fallback predictions used for {missing_pred_mask.sum():,} rows")


Segment Calibration Factors (adjustment to match actuals):
    segment  bias_factor
0      core     1.030320
1     large     0.957336
2  volatile     0.942329
Fallback predictions used for 0 rows


## Metric Interpretation for Leadership

Model quality is measured with complementary metrics so performance is understood from multiple perspectives.

| Metric | Interpretation |
|---|---|
| MAPE | Average percentage miss |
| WMAPE | Weighted miss focused on larger balances |
| R² | Consistency of predicted vs. actual variation |
| Median MAPE | Typical account-level miss, less influenced by extremes |

In [14]:
# VALIDATION METRICS: Evaluate model quality on holdout test set
# Compute error metrics to assess prediction accuracy before going to production

# Mean Absolute Percentage Error: average % error across all rows
mape = mean_absolute_percentage_error(test_df["ELIGIBLE_AMOUNT"], test_df["Prediction"])

# R² Score: explains how much of variance in actual values is explained by predictions
# 1.0 = perfect prediction, 0.0 = no better than mean, negative = worse than mean
r2 = r2_score(test_df["ELIGIBLE_AMOUNT"], test_df["Prediction"])

# Weighted Mean Absolute Percentage Error: focuses on larger accounts
# Sum of absolute errors / sum of actual values
wmape = (
    np.abs(test_df["ELIGIBLE_AMOUNT"] - test_df["Prediction"]).sum()
    / test_df["ELIGIBLE_AMOUNT"].sum()
)

# Account-level error metric: Absolute Percentage Error per row
# Useful for distribution analysis to understand typical account-level accuracy
test_df["APE"] = (
    np.abs(test_df["ELIGIBLE_AMOUNT"] - test_df["Prediction"])
    / test_df["ELIGIBLE_AMOUNT"]
)

# Print validation results
print(f"\n=== HOLDOUT VALIDATION RESULTS ===")
print(f"MAPE (Mean Abs Pct Error):  {mape:.2%}  [lower is better]")
print(f"R² (Explained Variance):    {r2:.3f}   [0-1 scale; higher is better]")
print(f"WMAPE (Weighted):           {wmape:.2%}  [focuses on large accounts]")
print(f"Median MAPE:                {test_df['APE'].median():.2%}  [typical account error]")



=== HOLDOUT VALIDATION RESULTS ===
MAPE (Mean Abs Pct Error):  36.52%  [lower is better]
R² (Explained Variance):    0.961   [0-1 scale; higher is better]
WMAPE (Weighted):           15.09%  [focuses on large accounts]
Median MAPE:                15.71%  [typical account error]


In [15]:
# FEATURE IMPORTANCE: Identify which inputs drive predictions
# Extract from fallback model (XGBoost); shows contribution of each feature to prediction
# Higher importance = feature is more influential in making predictions
feature_importance = (
    pd.DataFrame({"Feature": features, "Importance": model.feature_importances_})
    .sort_values("Importance", ascending=False)
)

print("\n=== TOP 10 MOST IMPORTANT FEATURES ===")
# Display feature importance dataframe with all features ranked
feature_importance



=== TOP 10 MOST IMPORTANT FEATURES ===


,Feature,Importance
2,lag_6,0.307656
4,rolling_3,0.261969
0,lag_1,0.145898
15,DELINQUENT_LOANS_TO_TOTAL_LOANS_RATIO,0.081170
5,rolling_6,0.042319
3,lag_12,0.037588
6,rolling_12,0.035263
1,lag_3,0.033901
14,LOANS_TO_ASSETS,0.022551
12,ASSETS_PER_MEMBER,0.020004


## Full-History Retrain

Segment models are retrained on the complete history before generating forward forecasts.
The segment design is fixed — only the training data is expanded.

In [16]:
# FULL-HISTORY RETRAIN: Use all data (train + test) to fit final production models
# Segment assignments remain the same (from training window only); expand training data
# This maximizes information available for final forecasts

# Build account profile using full dataset (not just training data)
# This recomputes segment anchors, means, and volatility with complete information
account_full_profile = (
    df.groupby(["ACCOUNT_ID", "ACCOUNT_NAME"], as_index=False)["ELIGIBLE_AMOUNT"]
    .agg(segment_anchor="median", segment_mean="mean", segment_std="std")
)

# Calculate coefficient of variation for full dataset (volatility metric)
account_full_profile["segment_cv"] = (
    account_full_profile["segment_std"].fillna(0.0)
    / account_full_profile["segment_mean"].replace(0, np.nan)
).fillna(0.0)

# Assign segments using same logic as before (based on full data thresholds)
# NOTE: We recalculate thresholds on full data, but segment assignments should be similar
q_size_large_full = account_full_profile["segment_anchor"].quantile(0.75)
q_vol_high_full = account_full_profile["segment_cv"].quantile(0.75)

# Apply segment assignment to full dataset
account_full_profile["segment"] = account_full_profile.apply(
    lambda r: assign_segment(r["segment_anchor"], r["segment_cv"]), axis=1
)

# Create segment map for full dataset
account_segment_map_full = account_full_profile[["ACCOUNT_ID", "segment"]].copy()
# Merge segment assignments with full dataset
df_with_segment = df.merge(account_segment_map_full, on="ACCOUNT_ID", how="left")

# Re-check minimum segment size (should be same, but being defensive)
# Count rows in each segment with full data
full_segment_counts = df_with_segment["segment"].value_counts()
# Identify segments below minimum threshold
small_full_segments = full_segment_counts[full_segment_counts < min_segment_rows].index.tolist()

# Consolidate small segments to core (safeguard against overfitting)
if small_full_segments:
    print(f"Consolidating {small_full_segments} to core due to insufficient rows")
    account_segment_map_full.loc[
        account_segment_map_full["segment"].isin(small_full_segments), "segment"
    ] = "core"
    # Re-merge with corrected segment assignments
    df_with_segment = df.merge(account_segment_map_full, on="ACCOUNT_ID", how="left")

# Build account-to-segment lookup for forecasting phase
# Maps account ID directly to segment assignment for quick lookup during prediction
account_segment_lookup = dict(
    zip(account_segment_map_full["ACCOUNT_ID"], account_segment_map_full["segment"])
)

# RETRAIN ALL MODELS ON FULL DATASET
# Clear previous models and retrain with all available data
segment_models = {}
for seg in segment_order:
    # Filter full dataset to only this segment
    seg_full = df_with_segment[df_with_segment["segment"] == seg].copy()

    # Skip segment if it has too few rows for reliable training
    if len(seg_full) < min_segment_rows:
        print(f"Skipping full retrain for '{seg}' due to low row count: {len(seg_full)}")
        continue

    # Use same hyperparameters as in initial training (ensures consistency)
    if seg == "volatile":
        # Volatile: RandomForest for robust ensemble predictions
        seg_model = RandomForestRegressor(
            n_estimators=400, max_depth=14, min_samples_leaf=2,
            random_state=42, n_jobs=-1
        )
    else:
        # Core and large: XGBoost for stable, interpretable predictions
        seg_model = XGBRegressor(
            objective="reg:squarederror", n_estimators=500, max_depth=6,
            learning_rate=0.03, subsample=0.8, colsample_bytree=0.8,
            random_state=42, n_jobs=-1
        )

    # Fit on full segment data (training + test combined)
    seg_model.fit(seg_full[features], seg_full["ELIGIBLE_AMOUNT"])
    # Store trained model
    segment_models[seg] = seg_model
    print(f"Full-history retrain complete for '{seg}' with {len(seg_full):,} rows")

# Retrain fallback model on all data (training + test)
fallback_model.fit(df[features], df["ELIGIBLE_AMOUNT"])
# Set as reference model for feature importance
model = fallback_model

print(f"\nFull dataset: {df['ACCOUNT_ID'].nunique():,} unique accounts")
print(f"Full dataset: {len(df):,} total rows ready for forecasting")

# JUMP CAP GUARDS: Limit first-month forecast movement to prevent unrealistic jumps
# Prevents model from making extreme predictions when real data transitions occur
# Defines maximum allowed percentage change from last actual to first forecast
segment_jump_caps = {
    "core": 0.40,        # Core: allow ±40% from last actual
    "large": 0.40,       # Large: stricter guard (40%)
    "volatile": 0.80     # Volatile: more flexible (80%) due to inherent volatility
}


Full-history retrain complete for 'core' with 6,408 rows
Full-history retrain complete for 'volatile' with 3,553 rows
Full-history retrain complete for 'large' with 3,323 rows

Full dataset: 873 unique accounts
Full dataset: 13,284 total rows ready for forecasting


## Recursive Forecasting Method

Future values are generated one month at a time for each account.

### Process overview
1. Start from the latest known account history
2. Add one new future month placeholder
3. Rebuild lag and rolling features for that month
4. Route each account to its segment model (or fallback)
5. Apply segment calibration and first-month jump guardrails
6. Append prediction and continue to the next month

This mirrors production conditions because real future values are unknown during forecasting.

In [17]:
# DYNAMIC FORECAST DATES - Simple approach
latest_date = df["ASSIGNMENT_DATE"].max()

# Start forecast next month (1 month after latest date)
forecast_start = latest_date + pd.DateOffset(months=1)
forecast_start = forecast_start.replace(day=1)

# Generate 12 months of forecast dates
forecast_dates = pd.date_range(start=forecast_start, periods=12, freq="MS")

print(latest_date)
print(f"Forecast dates: {forecast_dates[0]} to {forecast_dates[-1]}")


2026-05-01 00:00:00
Forecast dates: 2026-06-01 00:00:00 to 2027-05-01 00:00:00


In [18]:
# Function to prepare forecast features for recursive forecasting
# Rebuilds lag and rolling features for future months as new values are predicted
def prepare_forecast_features(history_df):
    """
    Prepare time-series features from history dataframe.
    This is called recursively during forecasting to update features with predicted values.
    """
    # Sort by account and date to ensure correct feature calculations
    history_df = history_df.sort_values(["ACCOUNT_ID", "ASSIGNMENT_DATE"])
    # Group by account for per-account feature engineering
    grp = history_df.groupby("ACCOUNT_ID")

    # Create lag features: shift eligible amounts by 1, 3, 6, 12 months
    # shift() moves values forward, creating lookback windows
    history_df["lag_1"]  = grp["ELIGIBLE_AMOUNT"].shift(1)
    history_df["lag_3"]  = grp["ELIGIBLE_AMOUNT"].shift(3)
    history_df["lag_6"]  = grp["ELIGIBLE_AMOUNT"].shift(6)
    history_df["lag_12"] = grp["ELIGIBLE_AMOUNT"].shift(12)

    # Create rolling average features: smooth trends using past values only
    # shift(1) + rolling() prevents future leakage by using only prior data
    history_df["rolling_3"]  = grp["ELIGIBLE_AMOUNT"].transform(lambda x: x.shift(1).rolling(3).mean())
    history_df["rolling_6"]  = grp["ELIGIBLE_AMOUNT"].transform(lambda x: x.shift(1).rolling(6).mean())
    history_df["rolling_12"] = grp["ELIGIBLE_AMOUNT"].transform(lambda x: x.shift(1).rolling(12).mean())

    # Calendar features: extract month and quarter for seasonal patterns
    history_df["month"]   = history_df["ASSIGNMENT_DATE"].dt.month
    history_df["quarter"] = history_df["ASSIGNMENT_DATE"].dt.quarter

    # Forward-fill NCUA columns so future rows carry the latest known values
    # NCUA data is quarterly, so we propagate it forward monthly
    for col in NCUA_COLS:
        if col in history_df.columns:
            history_df[col] = grp[col].transform(lambda x: x.ffill())

    return history_df


In [19]:
# RECURSIVE FORECASTING: Generate predictions one month at a time
# This mirrors production where future values are unknown until they occur
# Initialize list to collect monthly forecast dataframes
all_forecasts = []
# Start with complete historical data (train + test)
history = df.copy()
# Identify first forecast month for jump cap logic
first_fcst_date = forecast_dates.min()

# Loop through each forecast month
for forecast_date in forecast_dates:
    # List to collect new rows for this month (one per account)
    next_rows = []

    # Generate forecast for each active account
    for acct in history["ACCOUNT_ID"].unique():
        # Get all historical data for this account, sorted by date
        acct_hist = (
            history[history["ACCOUNT_ID"] == acct]
            .sort_values("ASSIGNMENT_DATE")
            .copy()
        )

        # Retrieve account name (constant across rows)
        acct_name = acct_hist["ACCOUNT_NAME"].iloc[-1]
        # Create new row by copying last historical row
        new_row = acct_hist.tail(1).copy()
        # Update date to forecast month and clear ELIGIBLE_AMOUNT (will predict)
        new_row["ASSIGNMENT_DATE"] = forecast_date
        new_row["ELIGIBLE_AMOUNT"] = np.nan

        # Combine history with new empty row to prepare features
        acct_future = pd.concat([acct_hist, new_row], ignore_index=True)
        # Rebuild features including lags/rolling averages based on combined data
        acct_future = prepare_forecast_features(acct_future)

        # Extract the new row with prepared features
        forecast_row = acct_future.tail(1)

        # Route to appropriate model based on account segment
        acct_segment = account_segment_lookup.get(acct, "core")
        seg_model = segment_models.get(acct_segment, fallback_model)

        # Generate prediction using segment model
        pred = seg_model.predict(forecast_row[features])[0]
        # Apply calibration factor for this segment
        cal_factor = segment_bias_factors.get(acct_segment, 1.0)
        pred = float(pred) * float(cal_factor)

        # FIRST-MONTH JUMP GUARDRAIL: Limit movement on first forecast month
        # Prevents model from making unrealistic jumps at transition point
        if forecast_date == first_fcst_date:
            # Get last known actual value from history
            last_actual_val = float(acct_hist["ELIGIBLE_AMOUNT"].iloc[-1])
            # Get jump cap for this segment
            jump_cap = segment_jump_caps.get(acct_segment, 0.40)
            # Calculate bounds: ±(jump_cap)% from last actual
            lower_cap = max(0.0, last_actual_val * (1 - jump_cap))
            upper_cap = last_actual_val * (1 + jump_cap)
            # Constrain prediction within bounds
            pred = min(max(pred, lower_cap), upper_cap)

        # Store prediction in new row
        new_row["ELIGIBLE_AMOUNT"] = pred
        new_row["ACCOUNT_NAME"] = acct_name
        # Add new row to list for this month
        next_rows.append(new_row)

    # Combine all accounts' new rows for this month
    next_month = pd.concat(next_rows, ignore_index=True)
    # Add this month's forecasts to results list
    all_forecasts.append(next_month)
    # Append to history so next month's features can reference this month's predictions
    history = pd.concat([history, next_month], ignore_index=True)

# Combine all forecast months into single dataframe
forecast_results = pd.concat(all_forecasts, ignore_index=True)
# Preview first few forecast rows
forecast_results.head()


,ACCOUNT_ID,ACCOUNT_NAME,ASSIGNMENT_DATE,ELIGIBLE_AMOUNT,TOTAL_ASSET_AMOUNT,CURRENT_MEMBER_COUNT,FULL_TIME_EQUIVALENT_EMPLOYEE_COUNT,ASSETS_PER_MEMBER,LOANS_TO_ASSETS,DELINQUENT_LOANS_TO_TOTAL_LOANS_RATIO,ANNUALIZED_ASSET_GROWTH,lag_1,lag_3,lag_6,lag_12,rolling_3,rolling_6,rolling_12,month,quarter
0,00100104,Avadian Credit Union,2026-06-01,9.232663e+06,1430149343,83228,270.0,17183.51,0.882593,0.003915,0.058063,8374914.0,6885230.0,7640610.0,6766558.0,7.743157e+06,7.456401e+06,8.174829e+06,5,2
1,00100340,Jefferson Credit Union,2026-06-01,5.240889e+05,92088175,8095,25.0,11375.93,0.597132,0.013827,-0.030582,643192.0,382970.0,475938.0,994757.0,5.407863e+05,6.103220e+05,6.726840e+05,5,2
2,00100641,City Credit Union,2026-06-01,7.720486e+05,20492086,1583,6.0,12945.10,0.752692,0.006102,0.223370,539896.0,539388.0,675004.0,775566.0,5.724287e+05,6.186877e+05,7.251118e+05,5,2
3,00101080,Fort McClellan Credit Union,2026-06-01,1.214269e+06,224739847,15331,66.5,14659.18,0.473702,0.010928,0.040382,988854.0,1017914.0,1379764.0,1709676.0,1.008227e+06,1.227298e+06,1.382980e+06,5,2
4,00101352,Listerhill Credit Union,2026-06-01,2.136244e+07,1457796387,91837,298.0,15873.74,0.821159,0.006875,0.075746,27944864.0,24431676.0,19747392.0,26009915.0,2.611264e+07,2.320201e+07,2.391575e+07,5,2


## Output Files and Reporting Use

The export section writes curated datasets for reporting and monitoring.

### Intended use of outputs
| File | Purpose |
|---|---|
| `eligible_amount_forecast_LDP_segmented_v2.csv` | Full forecast for operational consumption |
| `eligible_amount_forecast_with_bounds_segmented_v2.csv` | Forecast + bounds for risk-aware planning |
| `eligible_amount_history_forecast_with_bounds_segmented_v2.csv` | History + forecast for executive dashboards |
| `eligible_amount_history_forecast_with_bounds_segmented_v2_full_history.csv` | Full history + forecast for Power BI long-range visuals |

In [20]:
# Prepare historical data with record type identifier
# Copy full historical dataset
history_output = df.copy()
# Mark all rows as historical (not forecast)
history_output["Record_Type"] = "Historical"

# Mark forecast rows with record type identifier
forecast_results["Record_Type"] = "Forecast"

# Combine history and forecast into single dataframe
# This creates continuous timeline from first historical date through forecast period
final_df = pd.concat([history_output, forecast_results], ignore_index=True)
# Sort by account and date for clean ordering
final_df = final_df.sort_values(["ACCOUNT_ID", "ASSIGNMENT_DATE"])

# Export to CSV file for operational use
# File contains all historical data plus 12-month forecast
final_df.to_csv("eligible_amount_forecast_LDP_segmented_v2_dynamic.csv", index=False)
print("Forecast file exported successfully.")
print(final_df.shape)


Forecast file exported successfully.
(23760, 21)


## Confidence Bound Methodology

Uncertainty ranges are added around point forecasts using observed validation behavior.

### How bounds are built
- Compute account-level error behavior from the validation period
- Convert that into a percentage uncertainty factor (CU_MAPE)
- Apply this factor to each forecast point to form lower and upper bounds

This gives business users a planning range rather than a single deterministic value.

In [21]:
# Calculate account-level accuracy from validation period
# Median APE (Absolute Percentage Error) per account provides stable confidence estimate
# Groups by account and computes median error over holdout test months
account_accuracy = (
    test_df.groupby(["ACCOUNT_ID", "ACCOUNT_NAME"])["APE"]
    .median()
    .reset_index(name="CU_MAPE")
)

# Merge accuracy metrics into forecast results
# Each forecast row gets the account's typical error (CU_MAPE) from validation period
forecast_output = forecast_results.merge(account_accuracy, on=["ACCOUNT_ID", "ACCOUNT_NAME"], how="left")

# Cap confidence uncertainty at 70% (prevents extreme bounds for high-error accounts)
# This ensures bounds stay reasonable even for poorly predicted accounts
forecast_output["CU_MAPE"] = forecast_output["CU_MAPE"].clip(upper=0.70)

# Calculate lower bound: forecast * (1 - CU_MAPE)
# This represents the downside scenario based on account's typical error
forecast_output["Lower_Bound"] = forecast_output["ELIGIBLE_AMOUNT"] * (1 - forecast_output["CU_MAPE"])

# Calculate upper bound: forecast * (1 + CU_MAPE)
# This represents the upside scenario based on account's typical error
forecast_output["Upper_Bound"] = forecast_output["ELIGIBLE_AMOUNT"] * (1 + forecast_output["CU_MAPE"])


In [22]:
# Create dictionary with model performance metrics
# These metadata fields track model quality and version for audit trail
model_perf = {
    "MODEL_R2": float(r2),                          # R-squared: explained variance
    "MODEL_WMAPE": float(wmape),                    # Weighted mean absolute percentage error
    "MODEL_MAPE": float(mape),                      # Mean absolute percentage error
    "MODEL_MEDIAN_MAPE": float(test_df["APE"].median()),  # Median error per account
    "MODEL_VERSION": "segmented_v2",                # Version identifier for tracking
    "MODEL_RUN_DATE": pd.Timestamp.today().normalize()    # Date model was run
}

# Add model performance columns to all output dataframes
# This ensures every row includes the model metrics that produced it
for out_df in [final_df, forecast_output]:
    for k, v in model_perf.items():
        out_df[k] = v

# Prepare segment assignment for export
# Create segment mapping: ACCOUNT_ID -> SEGMENT
segment_map_export = account_segment_map_full[["ACCOUNT_ID", "segment"]].rename(
    columns={"segment": "SEGMENT"}
)

# Add segment assignment to all output dataframes
forecast_results = forecast_results.merge(segment_map_export, on="ACCOUNT_ID", how="left")
final_df = final_df.merge(segment_map_export, on="ACCOUNT_ID", how="left")


In [23]:
# Export forecast with bounds to CSV
# This file contains predictions with upper/lower confidence bounds for each forecast month
forecast_output.to_csv("eligible_amount_forecast_with_bounds_segmented_v2_dynamic.csv", index=False)
print("Forecast with bounds saved successfully.")

# Prepare combined history + forecast output for dashboard consumption
# This includes both actual historical data and forecast period with confidence bounds
final_output = final_df.merge(
    # Select only bound-related columns from forecast output
    forecast_output[["ACCOUNT_ID", "ACCOUNT_NAME", "ASSIGNMENT_DATE", "Lower_Bound", "Upper_Bound", "CU_MAPE"]],
    on=["ACCOUNT_ID", "ACCOUNT_NAME", "ASSIGNMENT_DATE"],
    how="left"
)

# Add model performance metadata to combined output
for k, v in model_perf.items():
    final_output[k] = v

# Export combined history + forecast to CSV
# This is the main file used for dashboards and strategic planning
final_output.to_csv(
    "eligible_amount_history_forecast_with_bounds_segmented_v2_dynamic.csv", index=False
)
print("Combined history + forecast file saved successfully.")


Forecast with bounds saved successfully.
Combined history + forecast file saved successfully.


## Simple Forecast Validation

Quick checks for:
- Holdout accuracy
- Tail error risk
- First-month forecast jump risk

In [24]:
# Prepare evaluation dataframe from test set
# Make a copy to avoid modifying original test data
eval_df = test_df.copy()
# Replace zeros with NaN to avoid division by zero in percentage calculations
denom = eval_df["ELIGIBLE_AMOUNT"].replace(0, np.nan)

# Calculate absolute percentage error for each row
# APE = |actual - predicted| / actual
eval_df["APE"] = (eval_df["ELIGIBLE_AMOUNT"] - eval_df["Prediction"]).abs() / denom
# Calculate absolute error (dollar amount, not percentage)
eval_df["AE"] = (eval_df["ELIGIBLE_AMOUNT"] - eval_df["Prediction"]).abs()

# Compute summary metrics from holdout test set
# Mean absolute percentage error across all rows
mape_val = eval_df["APE"].mean()
# Weighted mean absolute percentage error (focuses on larger accounts)
wmape_val = eval_df["AE"].sum() / eval_df["ELIGIBLE_AMOUNT"].sum()
# Median APE: typical account-level error (less influenced by outliers)
median_mape_val = eval_df["APE"].median()
# R² score: measure of explained variance
r2_val = r2_score(eval_df["ELIGIBLE_AMOUNT"], eval_df["Prediction"])
# 90th percentile APE: worst 10% of predictions
p90_ape_val = eval_df["APE"].quantile(0.90)
# Percentage of predictions with >100% error (very poor predictions)
pct_ape_gt_100_val = (eval_df["APE"] > 1.0).mean()

# Print validation scorecard for review
print("Validation Scorecard")
print(f"MAPE:          {mape_val:.2%}")
print(f"WMAPE:         {wmape_val:.2%}")
print(f"Median MAPE:   {median_mape_val:.2%}")
print(f"R2:            {r2_val:.3f}")
print(f"P90 APE:       {p90_ape_val:.2%}")
print(f"% APE > 100%:  {pct_ape_gt_100_val:.2%}")


Validation Scorecard
MAPE:          36.52%
WMAPE:         15.09%
Median MAPE:   15.71%
R2:            0.961
P90 APE:       53.52%
% APE > 100%:  3.17%


In [25]:
# First-month jump check: last actual vs. first forecast month
# Validates that jump cap guardrails are working correctly
# Identify the first forecast date
first_fcst_date = forecast_results["ASSIGNMENT_DATE"].min()

# Get last actual value for each account (from historical data)
# Sort by date and take the last row per account
last_actual = (
    df.sort_values(["ACCOUNT_ID", "ASSIGNMENT_DATE"])
    .groupby("ACCOUNT_ID", as_index=False)
    .tail(1)[["ACCOUNT_ID", "ELIGIBLE_AMOUNT"]]
    .rename(columns={"ELIGIBLE_AMOUNT": "LAST_ACTUAL"})
)

# Get first forecast month value for each account
# Filter to first forecast date and extract eligible amount
first_forecast = (
    forecast_results[forecast_results["ASSIGNMENT_DATE"] == first_fcst_date]
    [["ACCOUNT_ID", "ELIGIBLE_AMOUNT"]]
    .rename(columns={"ELIGIBLE_AMOUNT": "FORECAST_M1"})
)

# Merge last actual with first forecast for comparison
jump_check = last_actual.merge(first_forecast, on="ACCOUNT_ID", how="inner")

# Calculate absolute percentage difference between last actual and first forecast
# This shows how much the forecast "jumped" from the last known value
jump_check["ABS_PCT_DIFF"] = (
    (jump_check["FORECAST_M1"] - jump_check["LAST_ACTUAL"]).abs()
    / jump_check["LAST_ACTUAL"].replace(0, np.nan)
)

# Print jump analysis
print("First-month jump check")
# Show distribution of jumps: min, 25%, 50%, 75%, 90%, 95%, max
print(jump_check["ABS_PCT_DIFF"].describe(percentiles=[0.5, 0.9, 0.95]))
# Count how many accounts exceed 40% jump threshold
print(f"Count with jump > 40%: {(jump_check['ABS_PCT_DIFF'] > 0.40).sum():,}")


First-month jump check
count    873.000000
mean       0.128733
std        0.137772
min        0.000094
50%        0.084462
90%        0.310863
95%        0.400000
max        0.800000
Name: ABS_PCT_DIFF, dtype: float64
Count with jump > 40%: 25


## Post-Processing: Reattach Pre-Feature Historical Months for Power BI

This section does **not** retrain or modify the forecasting model. It only builds a supplemental export by adding back historical months that were dropped during feature-engineering `dropna`, so long-range visuals in Power BI can include the full active-account history.

In [ ]:

"""
OPTIONAL POST-PROCESSING: Add pre-feature-engineering historical months back to exports
This section does NOT retrain or modify the forecasting model.
It only builds a supplemental export by adding back historical months that were dropped during feature-engineering dropna,
so long-range visuals in Power BI can include the full active-account history.
"""

# Fetch raw LDP dataset from Snowflake instead of CSV
# Query the raw detail table to get all eligible amounts before feature engineering aggregation
conn = snowflake.connector.connect(
    user="*****************",
    account="************",
    authenticator="***********",     # Opens browser SSO login, no password needed
    warehouse="**************",
    database="*********",
    schema="*************",
    role="************"
)

# Query raw LDP table to get base eligible amounts
# Filters out null values and selects only the columns needed for post-processing
sql = """
SELECT
    ACCOUNT_NAME,
    ACCOUNT_ID,
    ASSIGNMENT_DATE,
    ELIGIBLE_AMOUNT
FROM S_CMG_BXU_ANALYST_DB_PROD.LX_PRESENTATION.LXU_AYX001065_D_CUSTOPT_MONTHLY_DATA_DETAIL_TB
WHERE ELIGIBLE_AMOUNT IS NOT NULL
ORDER BY ACCOUNT_ID, ASSIGNMENT_DATE
"""

# Execute query and fetch into pandas dataframe
cur = conn.cursor()
cur.execute(sql)
raw_scope = cur.fetch_pandas_all()
# Close connection after fetching data
conn.close()

# Convert ASSIGNMENT_DATE to datetime format for filtering operations
raw_scope["ASSIGNMENT_DATE"] = pd.to_datetime(raw_scope["ASSIGNMENT_DATE"])

# Apply same data filtering as main pipeline:
# 1. Convert dates to period format
month_index = raw_scope["ASSIGNMENT_DATE"].dt.to_period("M").drop_duplicates().sort_values()
# 2. Identify incomplete months (most recent two months)
incomplete_periods = month_index.tail(2)

# 3. Remove incomplete months from raw data
raw_scope = raw_scope[
    ~raw_scope["ASSIGNMENT_DATE"].dt.to_period("M").isin(incomplete_periods)
].copy()

# 4. Filter to active accounts only (those in latest month)
latest_month = raw_scope["ASSIGNMENT_DATE"].max()
active_accounts = raw_scope.loc[raw_scope["ASSIGNMENT_DATE"] == latest_month, "ACCOUNT_ID"].unique()
raw_scope = raw_scope[raw_scope["ACCOUNT_ID"].isin(active_accounts)].copy()

# 5. Aggregate to account-month level (sum eligible amounts, same as main pipeline)
raw_scope = (
    raw_scope.groupby(["ACCOUNT_ID", "ACCOUNT_NAME", "ASSIGNMENT_DATE"], as_index=False)
    ["ELIGIBLE_AMOUNT"].sum()
)

# Find rows in raw data that are NOT in final output
# These are the pre-feature-engineering months that were dropped
key_cols = ["ACCOUNT_ID", "ACCOUNT_NAME", "ASSIGNMENT_DATE"]
missing_history = raw_scope.merge(final_output[key_cols], on=key_cols, how="left", indicator=True)
# Keep only rows that exist in raw_scope but not in final_output
missing_history = missing_history[missing_history["_merge"] == "left_only"].drop(columns=["_merge"])
# Mark these rows as pre-feature historical data
missing_history["Record_Type"] = "Historical"

# Add segment assignment to missing rows (merge with segment map)
missing_history = missing_history.merge(segment_map_export, on="ACCOUNT_ID", how="left")

# Fill missing columns with NaN so schema matches final_output
for c in final_output.columns:
    if c not in missing_history.columns:
        missing_history[c] = np.nan

# Reorder columns to match final_output
missing_history = missing_history[final_output.columns]

# Combine missing pre-feature rows with main final_output
# This creates complete historical timeline from raw data through forecasts
final_output_full_history = pd.concat([missing_history, final_output], ignore_index=True)
# Sort by account and date for clean ordering
final_output_full_history = final_output_full_history.sort_values(["ACCOUNT_ID", "ASSIGNMENT_DATE"])

# Export full-history file for Power BI long-range visuals
full_history_path = "Full_History_NCUA_dynamic.csv"
final_output_full_history.to_csv(full_history_path, index=False)

print("Full-history export saved successfully.")
print(f"Rows added back for BI visuals: {len(missing_history):,}")
print(f"Total rows in full-history export: {len(final_output_full_history):,}")
print(f"Export path: {full_history_path}")
